In [53]:
import sys
from datetime import datetime
import numpy as np
def print_env():
    print('====================================', datetime.now().strftime('%Y-%m-%d %H:%M:%S'), '====================================')
    !cat /proc/cpuinfo  | grep 'model name'| uniq
    !cat /proc/cpuinfo  | grep 'cpu family' | uniq
    !cat /proc/cpuinfo  | grep 'model' | grep -v 'name' | uniq
    !cat /proc/cpuinfo  | grep 'MHz'| uniq
    !lscpu | grep '^CPU(s):'| uniq
    print('python:', sys.version)
    print('numpy:', np.__version__)
    print('====================================', '===================', '====================================')
print_env()  

==================================== 2026-10-09 17:46:31 ====================================
model name	: Intel Xeon Processor (Cascadelake)
cpu family	: 6
model		: 85
cpu MHz		: 2099.998
CPU(s):                                  8
python: 3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]
numpy: 2.3.5
==================================== =================== ====================================


In [54]:
import numpy as np
import tensorflow as tf
import time
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category = ConvergenceWarning) # suppress convergence warnings

(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data() # load MNIST train and test
n_features = X_train.shape[1] * X_train.shape[2]    # number of features: 784
n_train, n_test = X_train.shape[0], X_test.shape[0] # train: 60,000 / test: 10,000
X_train = X_train.reshape(n_train, n_features).astype('float32')  # one sample per row
X_test = X_test.reshape(n_test, n_features).astype('float32')     # one sample per row

In [55]:
# Rosenblatt's Perceptron Algorithm
def perceptron_algorithm(X, y, max_iter=100, verbose=False):
    y_ = np.where(y == 1, 1, -1)
    n_samples, n_features = X.shape
    w = np.zeros(n_features)
    b = 0.0
    for iter in range(max_iter):
        changed = False
        for i in range(n_samples):
            if y_[i] * (np.dot(w, X[i]) + b) <= 0:
                w += y_[i] * X[i]
                b += y_[i]
                changed = True
        if verbose:
            print(f'  epoch: {iter}/{max_iter}')
        if not changed:
            break
    return w, b

In [56]:
# normalize normal vector, adjust bias accordingly
def normalize(coef, intercept):
    coef_length = np.linalg.norm(coef)
    w = coef if coef_length == 0 else coef / coef_length
    b = intercept if coef_length == 0 else intercept / coef_length
    return w, b

In [57]:
# Oblique Decision Tree node (decision/leaf)
class ODT_node:
    nodes, decision_nodes, leaf_nodes, pos_leaves, neg_leaves = 0, 0, 0, 0, 0
    all_samples, pos_samples, neg_samples = 0, 0, 0
    def __init__(self, index=None, w=None, b=None, label=None, samples=0):
        self.index = index
        self.w = w
        self.b = b
        self.label = label
        self.samples = samples
        self.leaf = (label is not None)
        if self.leaf:
            ODT_node.leaf_nodes += 1
            ODT_node.all_samples += samples # only to checksum
            if label:
                ODT_node.pos_leaves += 1
                ODT_node.pos_samples += samples
            else:
                ODT_node.neg_leaves += 1
                ODT_node.neg_samples += samples
        else:
            ODT_node.decision_nodes += 1
        ODT_node.nodes += 1
    def add_neg_child(self, child):
        self.neg = child
    def add_pos_child(self, child):
        self.pos = child

In [58]:
# Oblique Decision Tree
class ODT:
    def __init__(self, root_node):
        self.root = root_node
        def visit(node): # explore the graph from root to set decision_nodes, leaf_nodes, all_nodes
            if node.leaf:
                return 0
            left = visit(node.neg)
            right = visit(node.pos)
            return 1 + left + right
        self.decision_nodes = visit(root_node)
        self.leaf_nodes = 1 + self.decision_nodes
        self.all_nodes = self.decision_nodes + self.leaf_nodes

In [59]:
# maximize: TP * TN + FP * FN by shifting bias
def optimize_bias(X, y, w):
    distances = X @ w.T # w normalized
    order = np.argsort(distances) # ascending
    d_sorted = distances[order] # least distance first
    y_sorted = y[order]
    del distances, order
    cand_bs = [] # candidates for: wx - cand_bs[i]
    for i in range(len(y) - 1):
        cand_bs.append(0.5 * (d_sorted[i] + d_sorted[i + 1])) # midpoints
    def div_score(b_param):
        preds = (d_sorted + b_param >= 0).astype('bool') # predictions: sign(w·x + b)
        dist = d_sorted + b_param # distances from boundary
        FTP = np.sum((y_sorted) & (preds) & (dist >= 0))
        FTN = np.sum((~y_sorted) & (~preds) & (dist <= 0))
        FFN = np.sum((y_sorted) & (~preds) & (dist <= 0))
        FFP = np.sum((~y_sorted) & (preds) & (dist >= 0))
        return FTP * FTN + FFN * FFP
    best_b = None
    best_score = -1
    for b in cand_bs:
        s = div_score(-b)
        if s > best_score:
            best_score = s
            best_b = -b
    del cand_bs, d_sorted, y_sorted
    return best_b



In [60]:
# splits, leaves, pos_leaves, neg_leaves = 0, 0, 0, 0
def split(X, y, method, coeffs, intercepts, index = 0):
    if y.shape[0] == 0: # empty set (should not happen)
        raise Exception('Empty set')
    if np.unique(y).size  == 1: # homogeneous set
        if y[0]: # positive leaf
            leaf = ODT_node(label=True, samples=X.shape[0]) # index=None, w=None, b=None, label=None, samples=0
        else: # negative leaf
            leaf = ODT_node(label=False, samples=X.shape[0]) # index=None, w=None, b=None, label=None, samples=0
        return leaf
    args = {} if '#' not in method else ast.literal_eval(method.split('#')[1])
    if method.startswith('lda'):
        if X.shape[0] == 2: # to avoid LDA ValueError: The number of samples must be more than the number of classes.
            w, b = perpendicular_bisector(X[0], X[1])
        else:
            model = LinearDiscriminantAnalysis(**args) # defaults: solver='svd' in {‘svd’, ‘lsqr’, ‘eigen’}
            model.fit(X, y)
            w = model.coef_[0]
            b = model.intercept_[0]
        w, b = normalize(w, b)
        b = optimize_bias(X, y, w)
    elif method.startswith('lr'):
        model = LogisticRegression(**args) # defaults: max_iter=100
        model.fit(X, y)
        w = model.coef_[0]
        b = model.intercept_[0]
        w, b = normalize(w, b)
        b = optimize_bias(X, y, w)
    elif method.startswith('svc'):
        model = SVC(kernel='linear', max_iter=1000, **args) # defaults: max_iter=-1 (no limit)
        model.fit(X, y)
        w = model.coef_[0]
        b = model.intercept_[0]
        w, b = normalize(w, b)
        b = optimize_bias(X, y, w)
    elif method.startswith('perceptron'):
        w, b = perceptron_algorithm(X, y, **args) # defaults: max_iter=100
        w, b = normalize(w, b)
        b = optimize_bias(X, y, w)
    elif method.startswith('sgd'):
        model = SGDClassifier(**args) # defaults: max_iter=1000, loss='hinge'
        model.fit(X, y)
        w = model.coef_[0]
        b = model.intercept_[0]
        w, b = normalize(w, b)
        b = optimize_bias(X, y, w)   
    elif method.startswith('lsvc'): # extremely slow on digit 8 with default max_iter=1000
        model = LinearSVC(**args) # defaults: max_iter=1000, loss='squared_hinge'
        # model = LinearSVC(max_iter=100, **args) # max_iter=100, loss='squared_hinge'
        model.fit(X, y)
        w = model.coef_[0]
        b = model.intercept_[0]
        w, b = normalize(w, b)
        b = optimize_bias(X, y, w)  
    else:
        raise(f'No method')

    index = len(coeffs)
    coeffs.append(w)
    intercepts.append(b)
    decision_node = ODT_node(index, w=w, b=b, samples=X.shape[0]) # index=None, w=None, b=None, label=None, samples=0

    distances = X @ w.T + b # w normalized
    pos_side_mask = (distances >= 0).astype('bool')
    neg_side_mask = (0 > distances).astype('bool')
    pos_side_X, pos_side_y = X[pos_side_mask], y[pos_side_mask]
    neg_side_X, neg_side_y = X[neg_side_mask], y[neg_side_mask]
    del distances, pos_side_mask, neg_side_mask,
    if len(pos_side_y) == len(y) or len(neg_side_y) == len(y):
        raise Exception('No split')

    neg_child = split(neg_side_X, neg_side_y, method, coeffs, intercepts)
    pos_child = split(pos_side_X, pos_side_y, method, coeffs, intercepts)

    decision_node.add_neg_child(neg_child)
    decision_node.add_pos_child(pos_child)

    del pos_side_X, pos_side_y, neg_side_X, neg_side_y
    return decision_node

In [61]:
def layer1(X, y, method, gamma=0):
    coeffs, intercepts = [], [] # last round results
    start = time.perf_counter()
    root = split(X, y, method, coeffs, intercepts, gamma)
    odt = ODT(root)
    elapsed_s = (time.perf_counter() - start)
    W = np.vstack(coeffs)
    b = np.array(intercepts)
    return W, b, odt

def leaf_patterns(root, l1, label):
    pattern = np.zeros(l1, dtype=np.int8)
    patterns = []
    def visit(node, label):
        if node.leaf:
            if node.label == label:
                patterns.append(pattern.copy())
            return
        pattern[node.index] = -1 # traverse negative child
        visit(node.neg, label)
        pattern[node.index] = +1 # traverse positive child
        visit(node.pos, label)
        pattern[node.index] = 0 # restore don't-care state when returning to parent
    visit(root, label)
    return np.array(patterns, dtype=np.int8)

def layer2(odt):
    W = leaf_patterns(odt.root, odt.decision_nodes, label=True)
    b = -(W == 1).sum(axis=1) + 0.5
    return W, b

def layer3(X): # OR
    W = np.ones(X.shape[1], dtype=np.int8)
    b = -0.5
    return W, b

# layer transformation with step activation    
def predict(X, W, b):
    return ((X @ W.T) + b >= 0).astype(int)    

In [62]:
class Binary_Classifier:
    def __init__(self, label, W1, b1, W2, b2, W3, b3):
        self.label = label
        self.W1 = W1
        self.b1 = b1
        self.W2 = W2
        self.b2 = b2
        self.W3 = W3
        self.b3 = b3
    def predict(self, X):
        X1 = predict(X, self.W1, self.b1)
        X2 = predict(X1, self.W2, self.b2)
        X3 = predict(X2, self.W3, self.b3).astype(bool)
        return X3 # True/False per sample
    def predict_sigmoid(self, X):
        X1 = predict_sigmoid(X, self.W1, self.b1)
        X2 = predict_sigmoid(X1, self.W2, self.b2)
        # X3 = predict_sigmoid(X2, self.W3, self.b3)
        # X3 = np.max(X2, ) # OR => if any of L2 neuron is confident, the result is confident
        X3 = np.max(X2, axis=-1, keepdims=True) 
        return X3
    def normalize(self):
        norms1 = np.linalg.norm(self.W1, axis=1)
        self.W1 = self.W1 / norms1[:, None]
        self.b1 = self.b1 / norms1     
        norms2 = np.linalg.norm(self.W2, axis=1)
        self.W2 = self.W2 / norms2[:, None]
        self.b2 = self.b2 / norms2   
        # W3: one hyperplane
        norm3 = np.linalg.norm(self.W3)
        self.W3 = self.W3 / norm3
        self.b3 = self.b3 / norm3     

In [63]:
class Multiclass_Classifier:
    def __init__(self, binary_classifiers):
        self.binary_classifiers = binary_classifiers
    def predict_all(self, X):
      return np.column_stack([classifier.predict(X) for classifier in self.binary_classifiers])
    def predict_all_sigmoid(self, X):
      return np.column_stack([classifier.predict_sigmoid(X) for classifier in self.binary_classifiers])
    def normalize(self):
        for classifier in self.binary_classifiers:
            classifier.normalize()

In [64]:
def evaluate(mc, X_train, y_train, X_test, y_test):
    # print(f'---------- Boolean Training set ----------')
    all_predictions = mc.predict_all(X_train) # bool matrix
    counts = all_predictions.sum(axis=1)
    x = np.argmax(all_predictions, axis=1)
    x[counts != 1] = -1
    print(f'  training accuracy: {100*np.sum(y_train == x)/X_train.shape[0]:.2f}% ({np.sum(y_train == x)}/{X_train.shape[0]})') # accuracy: 100.00%
    # print(f'---------- Boolean Test set ----------')
    all_predictions = mc.predict_all(X_test) # bool matrix
    counts = all_predictions.sum(axis=1)
    x = np.argmax(all_predictions, axis=1)
    x[counts != 1] = -1
    print(f'  test accuracy (strict): {100*np.sum(y_test == x)/X_test.shape[0]:.2f}% ({np.sum(y_test == x)}/{X_test.shape[0]})') # accuracy: 75.59%
    all_predictions = mc.predict_all(X_test)  # bool matrix
    correct = all_predictions[np.arange(len(y_test)), y_test]
    print(f'  test accuracy (light): {100*np.sum(correct)/len(y_test):.2f}% ({np.sum(correct)}/{len(y_test)})')

In [65]:
methods = ['lr', 'svc', 'perceptron'] # lr#{'max_iter':1000}

np.random.seed(42)

for method in methods:
    
    total_number_of_neurons = 0
    total_number_of_parameters = 0
    start = time.perf_counter()
    print(f'method: {method}')
    
    binary_classifiers = []
    layer_sizes, layer_times = [], []
    for digit in range(10):
        it_start = time.perf_counter()
        # print(f'DIGIT: {digit}')
        y = np.where(y_train == digit, True, False)
        # === layer 1 ===
        W1, b1, odt = layer1(X_train, y, method)
        X1 = predict(X_train, W1, b1)
        # === layer 2 ===
        W2, b2 = layer2(odt)
        X2 = predict(X1, W2, b2)
        # === layer 3 ===
        W3, b3 = layer3(X2)
        X3 = predict(X2, W3, b3).astype(bool)
        binary_classifiers.append(Binary_Classifier(digit, W1, b1, W2, b2, W3, b3))
        # === info ===
        it_elapsed_s = (time.perf_counter() - it_start)
        parameters = W1.shape[0] * (X_train.shape[1] + 1) + W2.shape[0] * (X1.shape[1] + 1) + 1 * (X2.shape[1] + 1)
        print(f'  {digit}: {W1.shape[0]}-{W2.shape[0]}-1, parameters: {parameters}, elapsed time: {it_elapsed_s:.2f}s')
        layer_sizes.append(f'{digit}: {W1.shape[0]}-{W2.shape[0]}-1')
        layer_times.append(f'{it_elapsed_s:.2f}s')
        total_number_of_neurons += W1.shape[0]+ W2.shape[0]+1
        total_number_of_parameters += parameters
    
    elapsed_s = (time.perf_counter() - start)
    print(f'Method: {method}')
    print(f'  Layer sizes: {layer_sizes}') 
    print(f'  Total number of neurons: {total_number_of_neurons}')
    print(f'  Construction times: {layer_times}')
    print(f'  Total construction time: {elapsed_s}')
    print(f'  Total number of neurons: {total_number_of_neurons}')
    # print(f'  Total number of neurons: {total_number_of_neurons}, Total number of parameters: {total_number_of_parameters}, Total elapsed time: {elapsed_s:.2f}s')
    mc = Multiclass_Classifier(binary_classifiers)
    evaluate(mc, X_train, y_train, X_test, y_test)

method: lr
  0: 11-6-1, parameters: 8714, elapsed time: 73.00s
  1: 11-6-1, parameters: 8714, elapsed time: 74.52s
  2: 22-12-1, parameters: 17559, elapsed time: 95.08s
  3: 30-16-1, parameters: 24063, elapsed time: 100.79s
  4: 17-9-1, parameters: 13517, elapsed time: 77.18s
  5: 27-14-1, parameters: 21602, elapsed time: 93.20s
  6: 15-8-1, parameters: 11912, elapsed time: 76.29s
  7: 17-9-1, parameters: 13517, elapsed time: 78.67s
  8: 51-26-1, parameters: 41414, elapsed time: 120.93s
  9: 42-22-1, parameters: 33939, elapsed time: 108.95s
Method: lr
  Layer sizes: ['0: 11-6-1', '1: 11-6-1', '2: 22-12-1', '3: 30-16-1', '4: 17-9-1', '5: 27-14-1', '6: 15-8-1', '7: 17-9-1', '8: 51-26-1', '9: 42-22-1']
  Total number of neurons: 381
  Construction times: ['73.00s', '74.52s', '95.08s', '100.79s', '77.18s', '93.20s', '76.29s', '78.67s', '120.93s', '108.95s']
  Total construction time: 898.626883732155
  Total number of neurons: 381
  training accuracy: 100.00% (60000/60000)
  test accuracy 